# Imaging-derived phenotypes with PIE

From a raw LONI imaging download to a table of imaging-derived phenotypes (IDPs) that joins the
tabular pipeline on `PATNO` / `EVENT_ID`, plus the diffusion, neuromelanin, DaTscan and FLAIR
pipelines built on the same T1 segmentation.

What this covers:

1. the two environments and the external tools each step needs;
2. what to download from LONI, and what the zips look like inside;
3. indexing, T1 selection, conversion and visit linking;
4. the resumable `python -m pie.imaging.run` CLI and its work directory;
5. the IDP table, and the join into `run_pipeline`;
6. the other modalities, labels and covariates, the cross-modality manifest and the QC galleries.

**No outputs are saved in this notebook, on purpose.** PPMI imaging is released under a data use
agreement, so a notebook in the repository must not carry participant IDs, LONI image IDs, scan
dates or per-subject values. Run it yourself to see numbers; keep what you print at the shape,
column and file-count level.

**Most cells here are templates.** Segmentation, conversion and the modality pipelines need
FastSurfer, dcm2niix, FSL, MRtrix3 or ANTs, real DICOM and hours of compute, so those cells sit
behind a `RUN_* = False` flag and print the command they would run. The cheap and safe things --
environment checks, the selection and naming rules on synthetic frames, the atlas provenance check,
reading a table you have already produced -- are real cells that run as they stand.

Depth lives in [imaging.md](../documentation/imaging.md),
[imaging_dwi.md](../documentation/imaging_dwi.md) and
[imaging_nm_datscan.md](../documentation/imaging_nm_datscan.md).

## Two environments

The imaging layer does not run in the tabular environment. `scripts/setup_imaging.sh` builds a
second one:

```bash
bash scripts/setup_imaging.sh            # NVIDIA default (cu128); `bash scripts/setup_imaging.sh cpu` for CPU torch
```

It needs [`uv`](https://docs.astral.sh/uv/), clones FastSurfer into `third_party/FastSurfer`, and
creates `venv_imaging/` (Python 3.12) holding FastSurfer's requirements -- torch for the chosen
backend, SimpleITK, scikit-image, MONAI -- plus `dcm2niix` (as a binary in `venv_imaging/bin/`),
pydicom, nibabel, DIPY, nilearn, ANTsPy and the modelling libraries.

Use `venv_imaging/bin/python` for everything in `pie.imaging`, and your tabular environment for
`pie.pipeline`. The two meet at one CSV: `fastsurfer_idps.csv`.

No FreeSurfer licence is required: only FastSurfer's segmentation stream is used, which gives
volumes but no cortical thickness or surface area.

In [ ]:
import os
import shutil
from importlib.util import find_spec
from pathlib import Path

REPO = Path.cwd() if (Path.cwd() / "pie").is_dir() else Path.cwd().parent   # run from the repo root or walkthroughs/
print("repo:", REPO.name, "| python:", ".".join(map(str, __import__("sys").version_info[:3])))

# Python packages the modality pipelines import
for pkg in ["nibabel", "pydicom", "SimpleITK", "skimage", "dipy", "nilearn", "ants", "monai", "torch", "pandas"]:
    print(f"  {pkg:<10}", "yes" if find_spec(pkg) else "NO")

# External binaries, and which step needs them
tools = {"dcm2niix": "every conversion", "topup": "dwi --fsl", "mrconvert": "dwi --fba / --denoise",
         "antsRegistration": "dwi_refine, nm_template"}
for tool, used_by in tools.items():
    print(f"  {tool:<16} {'found' if shutil.which(tool) else 'not on PATH':<12} ({used_by})")

### Where PIE looks for the heavy tools

Defaults are relative to the repository checkout, and every one of them can be overridden with an
environment variable read at import:

| Variable | Default | Used by |
|---|---|---|
| `PIE_DCM2NIIX` | `<repo>/venv_imaging/bin/dcm2niix` | all conversions |
| `PIE_FASTSURFER_HOME` | `<repo>/third_party/FastSurfer` | `run`, `fastsurfer` |
| `PIE_FASTSURFER_PYTHON` | `<repo>/venv_imaging/bin/python` | the interpreter FastSurfer is called with |
| `PIE_WEIGHTS_DIR` | `<repo>/third_party/weights` | `embed`, `cnn --pretrained` |
| `PIE_FASTSURFER_CPU_SECONDS` | 3600 | per-scan and stall budget when `--device` is not CUDA |
| `FSLDIR` | `~/fsl` if present | `dwi --fsl` (topup) |

The GPU is only used by FastSurfer inference and by `cnn` / `embed`. `run --device cpu` works and
gets hour-scale time budgets instead of the GPU's 90 s per scan, because CPU inference takes tens of
minutes per scan.

In [ ]:
from pie.imaging import convert, fastsurfer, embed

print("dcm2niix          :", convert.DCM2NIIX)
print("FastSurfer home   :", fastsurfer.FASTSURFER_HOME)
print("FastSurfer python :", fastsurfer.PYTHON)
print("weights root      :", embed.WEIGHTS_DIR)
print("GPU budget (s/scan, stall):", fastsurfer._budget("cuda"), "| CPU:", fastsurfer._budget("cpu"))
print("FSLDIR            :", os.environ.get("FSLDIR") or "(unset; ~/fsl used when present)")

## What to download from LONI

From the [LONI IDA](https://ida.loni.usc.edu/) advanced search, select the collection you want and
download **DICOM**. A download of the T1 collection used here arrives as four files, and PIE reads
all of them:

| File | What it is | Passed as |
|---|---|---|
| `MRI_First_Study.zip` | the images | `--zips` |
| `MRI_First_Study_dataset.zip` | the rest of the same collection when LONI splits it | `--zips` (order sets `protocol_phase`) |
| `MRI_First_Study_9_07_2026.csv` | the collection CSV LONI writes next to every download: one row per series with `Image Data ID`, `Subject`, `Group`, `Sex`, `Age`, `Visit`, `Description`, `Acq Date` | `--loni-csv` |
| `MRI_First_Study_IDA_Metadata.zip` | the `idaxs` XML of "Advanced Download": visit, research group, age, and the protocol terms (manufacturer, model, field strength, slice thickness, plane) | `--ida-metadata` |

SPECT comes as its own collection (`First_Study_SPECT.zip`, `First_Study_SPECT_dataset.zip`, and
their CSV and metadata zip). Put them all in `Imaging/`, next to the `PPMI/` study-data download
that supplies the visit table, the labels and the covariates.

Inside a zip, every series is a directory:

```
PPMI/<PATNO>/<SeriesDescription>/<yyyy-mm-dd_HH_MM_SS.0>/<IMAGE_ID>/*.dcm
```

PIE never unpacks the archive: it reads that listing, and extracts one series at a time into a
temporary directory when it converts. Keep the zips; they are the raw data, and the derived tree is
reproducible from them.

In [ ]:
IMAGING = REPO / "Imaging"                 # where the LONI downloads live
DERIVED = IMAGING / "derived"              # --work-dir of pie.imaging.run
PPMI_DIR = REPO / "PPMI"                   # the study-data download (tables, not images)

expected = ["MRI_First_Study.zip", "MRI_First_Study_dataset.zip", "MRI_First_Study_9_07_2026.csv",
            "MRI_First_Study_IDA_Metadata.zip", "First_Study_SPECT.zip"]
for name in expected:
    p = IMAGING / name
    print(f"  {name:<40}", "present" if p.exists() else "missing")
print("study-data tables:", "present" if (PPMI_DIR / "_Subject_Characteristics").is_dir() else "missing")
print("derived work dir :", "present" if DERIVED.is_dir() else "missing")

## Step 1: index the zips, pick one T1 per session

`index_zips(zip_paths, cache_csv=...)` walks the archive listings -- no extraction, no DICOM
parsing -- and returns one row per series: `zip`, `patno`, `series_desc`, `session`, `image_id`,
`n_files`, `bytes`, `member_prefix`, `session_date`. With `cache_csv` it is written once to
`index.csv` and reused, which matters because the listing of a 20 GB archive is not free.

`select_t1_series(index)` then picks one series per `(patno, session)`:

1. drop descriptions that are not a usable 3D T1 (localisers, calibration, coronal, phase, field
   maps, and every non-T1 modality);
2. prefer 3D/MPRAGE-family descriptions, penalise 2D/axial ones -- some sites label a two-frame
   axial T1 as the only "T1", and FastSurfer cannot use it;
3. penalise repeats (`repeat`, `rpt`, `_2`);
4. break ties by size in bytes, which also picks up single-file multi-frame DICOM.

Sessions dated 9999 (LONI masks some dates) survive with `date_masked=True`.

`probe_headers(sessions)` is the cheap way to get vendor and field strength for every session
straight from the zip, if you want scanner batches before converting anything.

In [ ]:
import pandas as pd

from pie.imaging.index import select_t1_series

# Synthetic listing: the ranking rules, without touching the archives.
demo = pd.DataFrame([
    dict(zip="MRI.zip", patno=1, series_desc="MPRAGE",        session="2000-01-01_09_00_00.0", image_id="IMG_A", n_files=176, bytes=9e7),
    dict(zip="MRI.zip", patno=1, series_desc="MPRAGE_Repeat", session="2000-01-01_09_00_00.0", image_id="IMG_B", n_files=176, bytes=9e7),
    dict(zip="MRI.zip", patno=1, series_desc="Localizer",     session="2000-01-01_09_00_00.0", image_id="IMG_C", n_files=3,   bytes=1e5),
    dict(zip="MRI.zip", patno=1, series_desc="AX_T1",         session="2001-01-01_09_00_00.0", image_id="IMG_D", n_files=2,   bytes=6e7),
    dict(zip="MRI.zip", patno=1, series_desc="3D_T1-weighted", session="2001-01-01_09_00_00.0", image_id="IMG_E", n_files=1,  bytes=2.5e7),
])
demo["member_prefix"] = "p/"
demo["session_date"] = pd.to_datetime(demo["session"].str[:10])

chosen = select_t1_series(demo)
print("one row per session:", chosen.shape)
print("chosen:", chosen["image_id"].tolist())          # the 3D non-repeat, and the 3D over the bigger 2D axial
print("columns:", chosen.columns.tolist())

In [ ]:
# The real index, if you have already built one. Shape and columns only.
index_csv = DERIVED / "index.csv"
if index_csv.exists():
    idx = pd.read_csv(index_csv, dtype={"image_id": str}, nrows=200_000)
    print("index.csv:", idx.shape, "| columns:", idx.columns.tolist())
else:
    print("no index.csv yet - it is written by the first run of pie.imaging.run")

## Step 2: conversion and visit linking

`convert_series(zip_path, member_prefix, patno, image_id, out_dir)` extracts one series to a
temporary directory, runs `dcm2niix -z y -b y`, and keeps
`nifti/<PATNO>/<IMAGE_ID>_T1w.nii.gz` plus the JSON sidecar. If dcm2niix splits a series it keeps
the largest volume. The sidecar fields worth having (vendor, model, field strength, TR/TE/TI, flip
angle, slice thickness) become columns of `scan_metadata.csv` and then of the IDP table, which is
what you harmonise on later. The extracted DICOM is deleted afterwards.

`link_sessions_to_events(sessions, ppmi_dir, max_months=3)` attaches the clinical visit. PPMI's
`Magnetic_Resonance_Imaging__MRI__*.csv` records the visit and its month (`INFODT`), while the DICOM
folder gives the exact date, so a session takes the `EVENT_ID` of the same month, else the nearest
within three months, else `UNK` (`months_off` records the gap). Only completed MRIs (`MRICMPLT == 1`)
count.

`--loni-csv` and `--ida-metadata` then override that link where their visit label maps cleanly
(`Baseline`/`BL` and `Screening`/`SC`), and add `loni_*` / `ida_*` columns. The joined result is
cached as `sessions.csv`; **delete that file if you add a CSV or metadata zip later**, or it will be
reused as it stands.

In [ ]:
import tempfile

from pie.imaging.link import link_sessions_to_events

# Synthetic PPMI visit table: month matching, and what "UNK" means.
with tempfile.TemporaryDirectory() as tmp:
    tables = Path(tmp) / "Imaging"
    tables.mkdir(parents=True)
    pd.DataFrame({"PATNO": [1, 1], "EVENT_ID": ["BL", "V04"], "INFODT": ["01/2000", "01/2001"],
                  "MRICMPLT": [1, 1]}).to_csv(tables / "Magnetic_Resonance_Imaging__MRI__2000.csv", index=False)
    sessions = pd.DataFrame({"patno": [1, 1, 1],
                             "session_date": pd.to_datetime(["2000-01-15", "2000-12-20", "2002-06-01"])})
    linked = link_sessions_to_events(sessions, tmp)

print(linked[["session_date", "EVENT_ID", "months_off"]].to_string(index=False))   # synthetic rows

## Step 3: the run CLI

One resumable command does index -> convert -> segment -> stats -> IDP table.

```bash
venv_imaging/bin/python -m pie.imaging.run \
    --zips Imaging/MRI_First_Study.zip Imaging/MRI_First_Study_dataset.zip \
    --ppmi-dir PPMI --work-dir Imaging/derived \
    --loni-csv Imaging/MRI_First_Study_9_07_2026.csv \
    --ida-metadata Imaging/MRI_First_Study_IDA_Metadata.zip \
    --workers 4 --threads 4
```

| Flag | Default | Why you would touch it |
|---|---|---|
| `--zips` | required | the archives; their order sets `protocol_phase` (1, 2, ...) |
| `--ppmi-dir` | `PPMI` | the study-data download, for the visit table |
| `--work-dir` | `Imaging/derived` | everything below is written here |
| `--workers` | 4 | CPU processes for conversion and statistics |
| `--threads` | 4 | CPU threads per FastSurfer call |
| `--chunk` | 20 | scans per GPU process; the models are loaded once per chunk |
| `--device` | `cuda` | `cpu`, or `cuda:1` to pick a card |
| `--loni-csv`, `--ida-metadata` | none | the visit labels and protocol terms above |
| `--priority` | none | a text file of PATNOs to process first: get your analysis cohort done before the rest |
| `--limit` | none | process at most N sessions this call -- the way to try five scans before committing a night |
| `--features-only` | off | rebuild `fastsurfer_idps.csv` from whatever has finished, without processing |

Pipeline shape: conversion runs in a CPU pool, FastSurferVINN runs on a chunk of scans in one GPU
process behind a file lock (one inference at a time, whatever `--workers` says), and N4 bias
correction plus partial-volume-corrected statistics run back on the CPU pool while the GPU takes the
next chunk.

In [ ]:
RUN_SEGMENTATION = False          # flip to True to actually process scans (hours, GPU)

cmd = [str(REPO / "venv_imaging/bin/python"), "-m", "pie.imaging.run",
       "--zips", str(IMAGING / "MRI_First_Study.zip"), str(IMAGING / "MRI_First_Study_dataset.zip"),
       "--ppmi-dir", str(PPMI_DIR), "--work-dir", str(DERIVED),
       "--loni-csv", str(IMAGING / "MRI_First_Study_9_07_2026.csv"),
       "--workers", "4", "--threads", "4", "--device", "cuda",
       "--limit", "5"]                                     # start small, then drop --limit

if RUN_SEGMENTATION:
    import subprocess
    subprocess.run(cmd, cwd=REPO, check=True)
else:
    print("template only; would run:\n ", " ".join(cmd))

### What lands in the work directory, and how to resume

| Path | What it is |
|---|---|
| `index.csv` | every series in the zips |
| `sessions.csv` | the chosen T1 per session, with `EVENT_ID` and the LONI/IDA columns |
| `scan_metadata.csv` | one row per converted scan: the dcm2niix sidecar fields |
| `nifti/<PATNO>/<IMAGE_ID>_T1w.nii.gz` | the converted T1 and its `.json` |
| `fastsurfer/<IMAGE_ID>/mri/` | `orig.mgz`, `orig_nu.mgz`, `mask.mgz`, `aparc.DKTatlas+aseg.deep.mgz` |
| `fastsurfer/<IMAGE_ID>/stats/aseg+DKT.stats` | the regional volumes; its presence means "done" |
| `fastsurfer/<IMAGE_ID>/scripts/` | the FastSurfer logs for that scan |
| `failures.csv` | one line per failed scan: PATNO, image id, message |
| `fastsurfer_idps.csv` | the wide IDP table, rebuilt on every call |

Resuming is just running the same command again: a session with `stats/aseg+DKT.stats` is skipped.
Interrupted scans are handled too -- a segmentation left half-written by a killed batch is deleted
and redone, a scan the batch skipped is retried in its own process, and one that still comes back
incomplete is logged rather than silently dropped.

**When a subject fails**, read `failures.csv` first. The common causes are a series that is not a
usable 3D volume (rejected before it can abort a GPU batch: not 3D, an axis under 40 voxels, or
voxels coarser than 2.5 mm), a scan FastSurfer itself refuses, and an out-of-memory kill. Re-running
retries every unfinished scan; nothing else is needed. If one scan poisons a batch, `--patnos`-style
targeting is not available here, but `--priority` plus `--limit` gets you the same effect.

In [ ]:
# Progress and failures, as counts. Never print the per-subject paths: they contain PATNOs.
fs_dir = DERIVED / "fastsurfer"
if fs_dir.is_dir():
    finished = sum(1 for _ in fs_dir.glob("*/stats/aseg+DKT.stats"))
    segmented = sum(1 for _ in fs_dir.glob("*/mri/aparc.DKTatlas+aseg.deep.mgz"))
    print(f"segmented: {segmented} | finished (stats written): {finished}")
    niftis = sum(1 for _ in (DERIVED / "nifti").glob("*/*_T1w.nii.gz")) if (DERIVED / "nifti").is_dir() else 0
    print("converted NIfTIs:", niftis)
else:
    print("no fastsurfer/ directory yet")

fail_csv = DERIVED / "failures.csv"
print("failure lines:", sum(1 for _ in open(fail_csv)) if fail_csv.exists() else 0)

## Step 4: the IDP table

`parse_stats` reads a FreeSurfer/FastSurfer `.stats` file into a flat dict, and `build_idp_table`
assembles one row per processed session:

| Columns | Meaning |
|---|---|
| `PATNO`, `EVENT_ID`, `IMAGEID`, `SCAN_DATE`, `protocol_phase` | identifiers; `protocol_phase` is which `--zips` archive the scan came from |
| `Manufacturer`, `ManufacturersModelName`, `MagneticFieldStrength`, `SoftwareVersions`, `InstitutionName`, `RepetitionTime`, `EchoTime`, `InversionTime`, `FlipAngle`, `SliceThickness` | scanner metadata, for harmonisation |
| `MaskVol`, `BrainSegVol`, ... | global measures: a measure whose short name ends in `Vol` keeps its name |
| `vol_<Structure>` | regional volume in mm^3, non-alphanumerics replaced by `_` (`vol_Left_Putamen`, `vol_ctx_lh_precuneus`, `vol_WM_hypointensities`) |
| `sum_<S>`, `asym_<S>` | left + right, and `(L - R) / (L + R)`, for putamen, caudate, pallidum, thalamus, hippocampus, amygdala, accumbens, lateral ventricle, cerebellar cortex and white matter, ventral DC |
| `sum_Ventricles` | lateral + inferior lateral + 3rd + 4th |

There is no eTIV (it needs a Talairach registration this stream does not do): use `MaskVol` as the
head-size normaliser. Asymmetry indices matter in PD, which is why they are precomputed.

In [ ]:
from pie.imaging.features import build_idp_table
from pie.imaging.fastsurfer import parse_stats

# A synthetic two-structure stats file, to show the naming rules end to end.
with tempfile.TemporaryDirectory() as tmp:
    stats = Path(tmp) / "IMG_A" / "stats"
    stats.mkdir(parents=True)
    (stats / "aseg+DKT.stats").write_text(
        "# Measure Mask, MaskVol, Mask Volume, 1500000.0, mm^3\n"
        "# ColHeaders Index SegId NVoxels Volume_mm3 StructName\n"
        "  1  12  5000  5000.0  Left-Putamen\n"
        "  2  51  4800  4800.0  Right-Putamen\n")
    print("parse_stats ->", parse_stats(stats / "aseg+DKT.stats"))

    sessions = pd.DataFrame([dict(patno=1, image_id="IMG_A", session_date="2000-01-01", EVENT_ID="BL",
                                  protocol_phase=1, Manufacturer="Siemens")])
    idp = build_idp_table(sessions, Path(tmp))

print("columns:", [c for c in idp.columns if c.startswith(("vol_", "sum_", "asym_")) or c == "MaskVol"])
print("asym_Putamen:", round(float(idp["asym_Putamen"].iloc[0]), 4))     # synthetic

In [ ]:
# Your real table, if it exists: shape and column families only.
idp_csv = DERIVED / "fastsurfer_idps.csv"
if idp_csv.exists():
    idps = pd.read_csv(idp_csv, nrows=5, low_memory=False)          # header is enough for the shape of the columns
    cols = pd.read_csv(idp_csv, nrows=0).columns
    print("columns:", len(cols))
    for prefix in ("vol_", "sum_", "asym_"):
        print(f"  {prefix:<6}", sum(c.startswith(prefix) for c in cols))
    print("identifier columns present:", [c for c in ("PATNO", "EVENT_ID", "IMAGEID", "SCAN_DATE") if c in cols])
else:
    print("no fastsurfer_idps.csv yet")

## Step 5: join it to the tabular pipeline

`run_pipeline(..., imaging_features=<csv>)`, or `--imaging-features <csv>` on
`python pie/pipeline.py`, adds the table as the `imaging` modality in stage 1. Numeric columns
become `imaging_<column>`; `IMAGEID`, `SCAN_DATE` and the text columns (the scanner strings) are
dropped, so keep a copy of the raw CSV if you want those for harmonisation.

The join is on `PATNO` and `EVENT_ID`, which is why the visit linking above matters: a scan whose
`EVENT_ID` is `UNK` matches no clinical visit and falls out of the merged frame.

Harmonise scanner effects **inside** the cross-validation folds, never on the full table before
splitting -- `endgame.preprocessing.ComBatHarmonizer(batch=..., covariates=[age, sex])` with the
scanner as the batch. For multi-modality work use `manifest.assemble_features` (below) instead of
this single CSV, and the per-modality batch columns it provides.

This cell runs in the **tabular** environment, not `venv_imaging`.

In [ ]:
RUN_PIPELINE_JOIN = False         # needs the tabular environment and the PPMI study-data download

if RUN_PIPELINE_JOIN and idp_csv.exists():
    from pie.pipeline import run_pipeline
    run_pipeline(
        data_dir=str(PPMI_DIR),
        output_dir=str(REPO / "output" / "with_imaging"),
        target_column="COHORT",
        leakage_features_path=str(REPO / "config" / "leakage_features.txt"),
        modalities=["subject_characteristics", "motor_assessments"],
        imaging_features=str(idp_csv),                 # CLI: --imaging-features
        n_models_to_compare=3,
        budget_time_minutes=20.0,
    )
else:
    print("template only - see walkthroughs/basic_classification.ipynb for the tabular side")

## The other modalities

Each one reuses the T1 you just produced: its segmentation is the anatomy, and `sessions.csv` says
which T1 belongs to which participant (the earliest session with finished statistics). They share a
resumable runner, so a killed job loses at most the subjects in flight, and each writes one CSV row
per subject with an `error` column instead of raising.

| Pipeline | Command | Needs | Output |
|---|---|---|---|
| Diffusion ([imaging_dwi.md](../documentation/imaging_dwi.md)) | `python -m pie.imaging.dwi --zips <DTI zips> --sessions ... --fastsurfer-dir ... --work-dir Imaging/derived/dwi` | dcm2niix, DIPY; `--fsl` adds FSL topup, `--fba` MRtrix3 | `dwi_features.csv`: FA/MD/free water per subcortical and nigral ROI |
| Neuromelanin ([imaging_nm_datscan.md](../documentation/imaging_nm_datscan.md)) | `python -m pie.imaging.nm --zips <full-MRI zips> ... --work-dir Imaging/derived/nm --keep-nifti` | dcm2niix, SimpleITK, nilearn | `nm_features.csv`: nigral contrast ratios |
| DaTscan ([imaging_nm_datscan.md](../documentation/imaging_nm_datscan.md)) | `python -m pie.imaging.datscan --index Imaging/derived/spect_index.csv --sessions ... --out-dir Imaging/derived/datscan_full` | pydicom, scikit-image | `datscan_sbr.csv`: striatal binding ratios from the raw projections |
| FLAIR | `python -m pie.imaging.flair --zips <full-MRI zips> ... --work-dir Imaging/derived/flair` | dcm2niix, SimpleITK | `flair_features.csv`: white-matter hyperintensity burden |

Flags shared by the DWI, NM and FLAIR runners: `--workers`, `--limit`, `--patnos` (a text file of
PATNOs), `--retry-errors` (redo the rows that recorded an error), `--keep-nifti` (keep the
per-subject volumes, which the QC galleries need), `--pid-file`.

DaTscan is the exception: it needs a `spect_index.csv` that you build from the SPECT zips
(`zip`, `member`, `patno`, `image_id`, `kind`, `frames`; only `kind == "TOMO"` rows are processed),
because PPMI ships raw tomographic projections rather than reconstructed volumes.

In [ ]:
RUN_MODALITIES = False            # each is hours of CPU across a cohort

py = str(REPO / "venv_imaging/bin/python")
common = ["--sessions", str(DERIVED / "sessions.csv"), "--fastsurfer-dir", str(DERIVED / "fastsurfer")]
jobs = {
    "dwi":   [py, "-m", "pie.imaging.dwi", "--zips", str(IMAGING / "<DTI download>.zip"), *common,
              "--work-dir", str(DERIVED / "dwi"), "--workers", "8", "--keep-nifti", "--fsl"],
    "nm":    [py, "-m", "pie.imaging.nm", "--zips", str(IMAGING / "MRI_First_Study.zip"), *common,
              "--work-dir", str(DERIVED / "nm"), "--workers", "4", "--keep-nifti"],
    "flair": [py, "-m", "pie.imaging.flair", "--zips", str(IMAGING / "MRI_First_Study.zip"), *common,
              "--work-dir", str(DERIVED / "flair"), "--workers", "6"],
}
if RUN_MODALITIES:
    import subprocess
    for name, job in jobs.items():
        subprocess.run(job, cwd=REPO, check=False)
else:
    for name, job in jobs.items():
        print(f"{name}: {' '.join(job[2:])}\n")

## Labels and covariates

`pie.imaging.labels` aligns outcomes to an MRI session. It reads the study-data tables, never the
images.

- `covariates(ppmi_dir)` -- cohort, enrolment, sex, birth month, handedness, `LRRK2` / `GBA` /
  `SNCA` / `APOE` and, when the table is present, the polygenic scores. Carrier coding is the trap:
  `0`, `0.0`, `"0"` and `"0.0"` all mean non-carrier whatever dtype the CSV parsed as, anything else
  is a carrier, and **blank stays NaN** -- an untested participant is not a control.
- `dat_labels(ppmi_dir, sessions, threshold=0.65, max_months=18)` -- the DaTscan closest to the
  scan: PPMI's own SBR columns, the visual read, and `dat_deficit_sbr`, the lowest putamen SBR below
  65 % of the age- and sex-expected value fitted on visually negative controls.
- `saa_labels(ppmi_dir, sessions, allow_unmatched=False)` -- CSF alpha-synuclein SAA at the scan's
  own visit, treating `SC` and `BL` as one occasion. It never borrows a later assay for an
  inconclusive visit, and conflicting calls at the chosen visit give no binary label.
  `allow_unmatched=True` reproduces the historical fallback for a clearly labelled sensitivity
  analysis, and records `saa_match` either way.

In [ ]:
from pie.imaging import labels

# Carrier coding across the dtypes a CSV can produce.
for values in [pd.Series([0, 1], dtype="int64"), pd.Series([0.0, 1.0, None]),
               pd.Series(["0", "G2019S", ""], dtype=object)]:
    print(f"{str(values.dtype):<8} {values.tolist()} -> {labels._carrier(values).tolist()}")

# saa_labels on a synthetic study-data tree: the match rules, not anyone's assay.
with tempfile.TemporaryDirectory() as tmp:
    bio = Path(tmp) / "Biospecimen"
    bio.mkdir(parents=True)
    pd.DataFrame({"PATNO": [1, 2, 2], "CLINICAL_EVENT": ["BL", "SC", "V06"],
                  "SAA_Status": ["Positive", "Negative", "Positive"],
                  "SAA_Type": ["Type1", None, "Type1"]}).to_csv(bio / "SAA_Biospecimen_Analysis_Results_2000.csv", index=False)
    sessions = pd.DataFrame({"patno": [1, 2], "image_id": ["IMG_A", "IMG_B"], "EVENT_ID": ["BL", "BL"]})
    saa = labels.saa_labels(tmp, sessions)

print("columns:", saa.columns.tolist())
print(saa[["SAA_EVENT_ID", "saa_positive", "saa_match", "saa_visit_concurrent"]].to_string(index=False))

## The atlas, and why it is checked

The nigral ROIs of the diffusion and neuromelanin pipelines come from the CIT168 atlas, bundled in
`pie/imaging/data/atlases/` as the authors' MNI152NLin2009cAsym projection with its checksum,
grid, label list and provenance.

This is worth a cell of its own because the obvious alternatives are wrong in ways nothing downstream
would reveal. Nilearn's `fetch_atlas_pauli_2017` deterministic file is in **native CIT168 space**,
not MNI152, and pushing it through an MNI-registered transform chain puts the substantia nigra in
the wrong place. Nilearn's default 2 mm `load_mni152_template` is a distinct 2009**a** image, so
registering to it and then mapping a 2009c atlas through the result mixes two spaces. So the
registration reference is bundled and checksummed next to the atlas, and the per-subject T1 -> MNI
cache carries a provenance sidecar (reference space and hash, the transform's own hash, the T1 and
mask hashes): `dwi.load_mni_cache` raises on an unverified or mismatched cache and leaves it on
disk, rather than reusing a transform fitted against a different template.

In [ ]:
from pie.imaging.atlases import cit168_metadata, cit168_mni2009c, cit168_provenance, mni2009c_template

atlas = cit168_mni2009c()                     # raises if space, checksum, grid or labels disagree
meta = cit168_metadata()
print("space:", meta["space"], "| version:", meta["version"], "| threshold:", meta["probability_threshold"])
print("atlas grid:", atlas.shape, "| labels:", len(meta["labels"]))
print("nigral labels:", [(i + 1, n) for i, n in enumerate(meta["labels"]) if n in ("SNc", "SNr")])
print("registration reference grid:", mni2009c_template().shape)
print("provenance columns written into NM rows:", sorted(cit168_provenance()))

## The manifest and the QC galleries

Each modality chose its own session, so the cross-modality table is not a plain join.
`build_manifest(derived_dir)` gives one row per subject: which session of each modality was used,
its date and interval to the T1 (`<mod>_days_from_t1`), the scanner batch (`<mod>_batch`), and a QC
flag (`<mod>_qc_pass`). If a modality row records a different T1 from the manifest's,
`<mod>_t1_mismatch` is set and that modality fails QC.

`assemble_features(derived_dir)` then produces the wide table -- FastSurfer IDPs plus `dat_*`,
`dwi_*`, `nm_*`, `flair_*` -- with every value of a QC-failed modality blanked, so a failed left
side cannot let a plausible right side through. `feature_blocks(df.columns)` groups the columns by
modality, which is what block-wise harmonisation and stacking need.

The QC rules live in `manifest.QC`, one per modality, so the galleries and any study agree on what
"pass" means:

| Modality | Passes when |
|---|---|
| `t1` | the key subcortical labels are non-empty (an empty label means FastSurfer failed) |
| `dat` | \|`reg_metric`\| >= 0.4 and >= 100 striatal label voxels |
| `dwi` | max motion < 6 mm, >= 3 SN voxels per side, white-matter median FA > 0.25 |
| `nm` | >= 20 SN voxels per side, slab coverage >= 0.5, repeat motion < 3 mm, a homogeneous reference |
| `flair` | registration metric < -0.2, plausible white-matter volume, non-zero MAD |

Use the per-modality `*_batch` columns for harmonisation: diffusion and neuromelanin features
harmonised by the *T1* scanner is a real mistake that this table exists to prevent.

In [ ]:
from pie.imaging.manifest import QC, assemble_features, build_manifest, feature_blocks

print("QC rules:", sorted(QC))

# A synthetic derived directory, so the manifest logic is visible without a cohort.
with tempfile.TemporaryDirectory() as tmp:
    d = Path(tmp)
    pd.DataFrame({"PATNO": [1, 2], "IMAGEID": ["IMG_A", "IMG_B"], "SCAN_DATE": ["2000-01-01"] * 2,
                  "vol_Left_Putamen": [5000.0, 5100.0], "vol_Right_Putamen": [4800.0, 4900.0],
                  "vol_Left_Caudate": [3500.0, 3600.0], "vol_Right_Caudate": [3400.0, 3500.0],
                  "vol_Left_Thalamus": [7000.0, 7100.0], "vol_Right_Thalamus": [6900.0, 7000.0],
                  "MaskVol": [1.5e6] * 2}).to_csv(d / "fastsurfer_idps.csv", index=False)
    (d / "dwi").mkdir()
    pd.DataFrame({"patno": [1, 2], "error": ["", ""], "motion_mm_max": [1.0, 9.0],      # subject 2 moved
                  "n_sn_l": [40, 40], "n_sn_r": [40, 40], "fa_wm_median": [0.4, 0.4],
                  "manufacturer": ["Siemens", "GE"], "shells": ["1000", "700 1000 2000"],
                  "fw_method": ["singleshell_prior", "multishell_nls"],
                  "sn_posterior_mean_fw": [0.30, 0.42], "acquisition_date": ["2000-01-03", "2000-01-05"],
                  "fs_image_id": ["IMG_A", "IMG_B"]}).to_csv(d / "dwi" / "dwi_features.csv", index=False)

    man = build_manifest(d)
    feats = assemble_features(d)

print("manifest columns:", man.columns.tolist())
print(man[["PATNO", "dwi_batch", "dwi_qc_pass", "dwi_days_from_t1"]].to_string(index=False))   # synthetic
print("blocks:", {k: v for k, v in feature_blocks(feats.columns).items() if v})
print("QC-failed values blanked:", feats["dwi_sn_posterior_mean_fw"].isna().tolist())

### Looking at the images

Numbers do not tell you that a registration slipped. `pie.imaging.qc` renders one PNG per subject --
three orthogonal views with the ROI contours drawn on the image the pipeline actually used -- plus a
contact sheet of the first 16:

```bash
venv_imaging/bin/python -m pie.imaging.qc --work-dir Imaging/derived/dwi --modality dwi \
    --out Imaging/derived/qc/dwi --n 40 --worst reg_b0_t1_mi
```

`--worst` sorts by a QC column in its failure direction, so you review the worst 40 rather than a
random 40; without it you get a random sample. What to look for: for `dwi`, the red substantia-nigra
contours sitting on the dark nigral band rather than in the peduncle; for `nm`, the refined SN (red)
on the bright band and the reference ring (lime) inside the brainstem; for `flair`, the lesion mask
following the hyperintensities rather than the ventricle edge; for `datscan`, the striatal labels on
the hot spots.

The galleries need the per-subject volumes, so run the modality with `--keep-nifti`. `datscan`
additionally needs `--sessions` and `--fastsurfer-dir`, because it re-applies the stored transform.

In [ ]:
RUN_QC = False                     # cheap per subject, but it reads the saved volumes

qc_cmd = [py, "-m", "pie.imaging.qc", "--work-dir", str(DERIVED / "nm"), "--modality", "nm",
          "--out", str(DERIVED / "qc" / "nm"), "--n", "40", "--worst", "sn_slab_coverage"]
if RUN_QC:
    import subprocess
    subprocess.run(qc_cmd, cwd=REPO, check=True)
else:
    print(" ".join(qc_cmd[2:]))

gallery = DERIVED / "qc" / "nm" / "gallery.png"
print("gallery present:", gallery.exists())

## Cost, hardware and disk

| Step | Order of magnitude |
|---|---|
| Indexing the zips | minutes for the listing, once; cached in `index.csv` |
| Conversion (dcm2niix) | seconds per scan, parallel over `--workers` |
| FastSurfer segmentation, GPU | ~30-60 s per scan on an RTX 2080; this is the bottleneck |
| FastSurfer segmentation, CPU | tens of minutes per scan; budgets rise to an hour per scan automatically |
| N4 + statistics | a few minutes per scan, on the CPU pool, overlapped with the GPU |
| DWI per subject | minutes; `--fsl` topup adds ~3-7 min, `--fba` ~2-3 min |
| Neuromelanin, FLAIR, DaTscan per subject | seconds to a few minutes |
| Disk | ~20 GB per 1,000 scans for NIfTI plus segmentations, on top of the zips |

FastSurferVINN wants about 6 GB of GPU memory, and PIE serialises inference with a file lock so
`--workers 8` does not put eight models on one card. If the machine is shared, keep `--workers` and
`--threads` modest: the conversion and statistics pool is what competes with everything else.

Plan the first run as `--limit 5`, confirm `fastsurfer_idps.csv` looks right, then start the cohort
with `--priority` pointing at the participants your analysis actually needs.

## Where to go next

- **The full reference.** [imaging.md](../documentation/imaging.md) documents every module, flag and
  output column, including the shared runner, the bundled atlas, the ZIP index cache, the DICOM
  header audit and the verified staging helpers.
- **Diffusion in depth.** [imaging_dwi.md](../documentation/imaging_dwi.md): run assembly and the
  acquisition key, gradient rotation, free-water fitting, the nigrostriatal tract measures, and the
  opt-in tensor and eddy-correction safeguards.
- **Neuromelanin and DaTscan.** [imaging_nm_datscan.md](../documentation/imaging_nm_datscan.md):
  repeat selection, the slab-to-T1 registration, the template pipeline, and the SPECT
  reconstruction and SBR chain.
- **Whole-image models.** `pie.imaging.cnn` (a grouped out-of-fold 3D CNN baseline) and
  `pie.imaging.embed` (frozen embeddings from pretrained open-weight brain MRI models) answer
  "would a network see something the region features miss?".
- **Turning measurements into a result.** [experiment.md](../documentation/experiment.md) for
  cohort rules, nested selection that never touches its test partition, and provenance manifests;
  [basic_classification.ipynb](basic_classification.ipynb) for the tabular pipeline these features
  join.